In [1]:
import torch
from PIL import Image

from share4v.constants import (DEFAULT_IM_END_TOKEN, DEFAULT_IM_START_TOKEN,
                               DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX)
from share4v.conversation import SeparatorStyle, conv_templates
from share4v.mm_utils import get_model_name_from_path, tokenizer_image_token, KeywordsStoppingCriteria
from share4v.model.builder import load_pretrained_model
from share4v.utils import disable_torch_init
from transformers import GenerationConfig

/home/lcorti/miniconda3/envs/lmms/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_image(image_file):
    return Image.open(image_file).convert('RGB')

def format_query(q, model):
    if model.config.mm_use_im_start_end:
        qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + \
            DEFAULT_IM_END_TOKEN + '\n' + q
    else:
        qs = DEFAULT_IMAGE_TOKEN + '\n' + q
    
    return qs

def set_conv_mode(model_name):
    if 'llama-2' in model_name.lower():
        conv_mode = "share4v_llama_2"
    elif "v1" in model_name.lower():
        conv_mode = "share4v_v1"
    elif "mpt" in model_name.lower():
        conv_mode = "mpt"
    else:
        conv_mode = "share4v_v0"
    
    return conv_mode

def get_conv_template(conv_mode):
    return conv_templates[conv_mode].copy()

def get_stop_str(conv):
    return conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2

def get_stopping_criteria(input_ids, stop_str, tokenizer):
    keywords = [stop_str]
    stopping_criteria = KeywordsStoppingCriteria(keywords, tokenizer, input_ids)
    return stopping_criteria

def decode_tokens(input_ids, output_ids, tokenizer, stop_str):
    input_token_len = input_ids.shape[1]
    n_diff_input_output = (input_ids != output_ids[:, :input_token_len]).sum().item()

    if n_diff_input_output > 0:
        print(f'[Warning] {n_diff_input_output} output_ids are not the same as the input_ids')
    outputs = tokenizer.batch_decode(output_ids[:, input_token_len:], skip_special_tokens=True)[0]
    outputs = outputs.strip()

    if outputs.endswith(stop_str):
        outputs = outputs[:-len(stop_str)]
    outputs = outputs.strip()
    
    return outputs

In [3]:
def get_gen_config(do_sample, use_cache, temperature, max_new_tokens):
    gen_config = GenerationConfig.from_dict({
        'do_sample': do_sample,
        'temperature': temperature,
        'use_cache': use_cache,
        'max_new_tokens': max_new_tokens
    })
    '''
    If scores are needed:
    'output_scores': True,
    'return_dict_in_generate': True
    '''
    return gen_config
def gen_response(model, input_ids, image_tensor, gen_config, stopping_criteria):
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=image_tensor,
            generation_config=gen_config,
            stopping_criteria=[stopping_criteria])
    return output_ids

In [4]:
# Model loading parameters
model_path = "Lin-Chen/ShareGPT4V-7B"
model_base = None
model_name = get_model_name_from_path(model_path)
conv_mode = set_conv_mode(model_name)

# Generation config
do_sample = False
temperature = 0.1
max_new_tokens = 512
use_cache = True
gen_config = get_gen_config(do_sample,
                            use_cache,
                            temperature,
                            max_new_tokens)

# Load model
disable_torch_init() # This is part of the original implementation

tokenizer, model, image_processor, _ = load_pretrained_model(
    model_path=model_path,
    model_base=model_base,
    model_name=model_name,
    device_map='auto',
)
model.half()
print("loaded")

/home/lcorti/miniconda3/envs/lmms/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/home/lcorti/miniconda3/envs/lmms/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.79s/it]
/home/lcorti/miniconda3/envs/lmms/lib/python3.12/site-packages/transformers/generation/configuration_util

Load vision tower from Lin-Chen/ShareGPT4V-7B_Pretrained_vit-large336-l12
loaded


In [5]:
def add_conv_step(conv, user_q, prev_resp=None):
    # conv.roles[0] is user
    # conv.roles[1] is assistant
    if prev_resp:
        # Remove last ['Assistant', None] used at the previous generation step
        conv.messages.pop(-1)
        # Add assistant's generated response
        conv.append_message(conv.roles[1], prev_resp)
    conv.append_message(conv.roles[0], user_q)
    conv.append_message(conv.roles[1], None)
    return conv

In [22]:
# Main
user_q = "Discuss the historical impact and the significance of this painting in the art world."
q = format_query(user_q, model)
conv = get_conv_template(conv_mode)

# Prepare image input
image = load_image("/home/lcorti/lmm_diagnosis/datasets/llava-bench/imgs/006.jpg")
image_tensor = image_processor.preprocess(image, return_tensors='pt')['pixel_values'].half().cuda()

# Initial question
conv = add_conv_step(conv, q)
prompt = conv.get_prompt()
stop_str = get_stop_str(conv)

# Prepare input ids
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
stopping_criteria = get_stopping_criteria(input_ids, stop_str, tokenizer)

In [23]:
# Generation step
output_ids = gen_response(model, input_ids, image_tensor, gen_config, stopping_criteria)
response = decode_tokens(input_ids, output_ids, tokenizer, stop_str)
print(response)

# del image_tensor
# torch.cuda.empty_cache()

/home/lcorti/miniconda3/envs/lmms/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


The painting of a woman with long hair, often referred to as the Mona Lisa, is a famous artwork created by the Italian artist Leonardo da Vinci. The painting is a portrait of a woman, and it is considered one of the most iconic and influential works of art in the history of Western art. The Mona Lisa is known for its enigmatic smile and the subtle play of light and shadow, which adds depth and dimension to the painting.

The painting's historical impact lies in its ability to capture the essence of human beauty and the subtle nuances of human expression. The Mona Lisa has been a subject of fascination and admiration for centuries, and its enigmatic smile has sparked countless interpretations and theories about the woman's identity and the meaning behind her smile.

The painting's significance in the art world is further emphasized by its influence on other artists and its impact on the development of portraiture and the use of light and shadow in art. The Mona Lisa has inspired countle

Ask for explanation and then ask to format

In [24]:
expl_q = "Review the given answer and carefully analyze the image. "\
    "Identify and describe the key visual elements and concepts in the image that directly support your reasoning and answer. "\
    "For each point, clearly explain how that visual element contributes to the answer, and ensure every point is directly linked to the reasoning behind the answer.\n"\
    "Present your rationale in a structured, bullet-point format with the following guidelines:\n"\
    "- Each bullet point must begin with the * character.\n"\
    "- Each point should clearly explain the connection between a specific visual element and the given answer.\n"\
    "- Avoid general descriptions of the image; focus on visual details that are relevant to the answer.\n"\
    "- Keep each point concise and ensure it directly addresses the reasoning behind the answer."

conv = add_conv_step(conv, expl_q, prev_resp=response)
stop_str = get_stop_str(conv)
prompt = conv.get_prompt()

# Tokenize new input
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
stopping_criteria = get_stopping_criteria(input_ids, stop_str, tokenizer)
output_ids = gen_response(model, input_ids, image_tensor, gen_config, stopping_criteria)
expl_res = decode_tokens(input_ids, output_ids, tokenizer, stop_str)
print(expl_res)

* The painting features a woman with long hair, which is a common characteristic of portraiture in the Renaissance period.
* The woman's enigmatic smile is a key visual element that adds depth and intrigue to the painting.
* The subtle play of light and shadow in the painting is a hallmark of the Renaissance art style, and it adds a sense of realism and three-dimensionality to the portrait.
* The painting's enduring appeal and its status as a masterpiece are a testament to the enduring power of art to captivate and inspire.
* The painting has inspired countless artists to explore the human form and the subtle play of light and shadow in their works.
* The painting's historical impact lies in its ability to capture the essence of human beauty and the subtle nuances of human expression.
* The painting's significance in the art world is further emphasized by its influence on the development of portraiture and the use of light and shadow in art.


In [25]:
struct_expl_q = "Carefully analyze each point in the list above and extract distinct pairs of entities and their relationships. "\
    "For each point, identify exactly two distinct entities at a time, along with a detailed and context-specific relationship between them (e.g., action, spatial position, interaction, causality, part-whole relationship, etc.). "\
    "For each extraction, you must use the following format exactly: (Entity: [entity_1], Relationship: [relationship], Entity: [entity_2]).\n"\
    "Additional guidelines:\n"\
    "- Both entities must always be present: Ensure that each extraction contains two distinct entities in the format provided.\n"\
    "- Do not include references to the image itself (e.g., 'image,' 'picture,' 'photo') as an entity.\n"\
    "- Avoid overly simplistic relationships (e.g., 'located in' or 'visible in'). "\
        "Focus on specific interactions or connections, such as 'forms,' 'comprises,' 'is part of,' 'affects,' or 'is near.'\n"\
    "- Focus on distinct entity pairs, avoiding duplicates across all points.\n"\
    "- If a point contains multiple entities or relationships, break them down into separate pairs.\n"\
    "- Entities can refer to physical objects, people, animals, abstract concepts, etc.\n"\
    "- Relationships should reflect more complex interactions or spatial/temporal arrangements.\n"\
    "- Ensure the relationship clearly reflects how the entities are connected in context. "\
        "Consider relationships like causality, containment, or part-whole structure.\n"\
    "- Prioritize clarity and contextual relevance in your extractions.\n"\
    "- Each extraction must contain two distinct entities; avoid incomplete extractions with only one entity.\n"\
    "Example:\n"\
    "- Original sentence: 'A dog is playing with a ball near the tree.'\n"\
    "- Output:\n"\
    "  - (Entity: dog, Relationship: playing with, Entity: ball)\n"\
    "  - (Entity: dog, Relationship: near, Entity: tree)\n"\
    "  - (Entity: tree, Relationship: part of, Entity: landscape)"

conv = add_conv_step(conv, struct_expl_q, prev_resp=expl_res)
stop_str = get_stop_str(conv)
prompt = conv.get_prompt()

# Tokenize new input
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
stopping_criteria = get_stopping_criteria(input_ids, stop_str, tokenizer)
output_ids = gen_response(model, input_ids, image_tensor, gen_config, stopping_criteria)
struct_res = decode_tokens(input_ids, output_ids, tokenizer, stop_str)
print(struct_res)

* The painting features a woman with long hair, which is a common characteristic of portraiture in the Renaissance period. (Entity: Renaissance period, Relationship: characterized by, Entity: portraiture)
* The woman's enigmatic smile is a key visual element that adds depth and intrigue to the painting. (Entity: enigmatic smile, Relationship: adds, Entity: depth and intrigue)
* The subtle play of light and shadow in the painting is a hallmark of the Renaissance art style, and it adds a sense of realism and three-dimensionality to the portrait. (Entity: Renaissance art style, Relationship: hallmark, Entity: play of light and shadow, Relationship: adds, Entity: realism and three-dimensionality)
* The painting's enduring appeal and its status as a masterpiece are a testament to the enduring power of art to captivate and inspire. (Entity: enduring appeal, Relationship: testament to, Entity: masterpiece)
* The painting has inspired countless artists to explore the human form and the subtle 

Old prompt

In [24]:
struct_expl_q = "Carefully analyze each point in the list and extract distinct entities and their relationships. "\
    "For each point, identify only two entities at a time, along with the relationship between them (e.g., action, spatial position, interaction, etc.). "\
    "For each extraction, use the following format: (Entity: [entity_1], Relationship: [relationship], Entity: [entity_2]).\n"\
    "Additional guidelines:\n"\
    "- Do not include references to the image itself (e.g., 'image', 'picture', 'photo') as an entity.\n"\
    "- Focus on distinct entity pairs, avoiding duplicates across all points.\n"\
    "- If a point contains multiple entities or relationships, break them down into separate pairs.\n"\
    "- Entity can refer to physical objects, people, animals, abstract concepts, etc.\n"\
    "- Relationships should capture interactions (e.g., actions, comparisons, associations) or spatial/temporal arrangements.\n"\
    "- Ensure the relationship clearly reflects how the entities are connected in context.\n"\
    "- Prioritize simplicity and clarity in your extractions.\n"\
    "Example:\n"\
    "- Original sentence: 'A dog is playing with a ball near the tree.'\n"\
    "- Output:\n"\
    "   - (Entity: dog, Relationship: playing with, Entity: ball)\n"\
    "   - (Entity: dog, Relationship: near, Entity: tree)"

conv = add_conv_step(conv, struct_expl_q, prev_resp=expl_res)
stop_str = get_stop_str(conv)
prompt = conv.get_prompt()

# Tokenize new input
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
stopping_criteria = get_stopping_criteria(input_ids, stop_str, tokenizer)
output_ids = gen_response(model, input_ids, image_tensor, gen_config, stopping_criteria)
struct_res = decode_tokens(input_ids, output_ids, tokenizer, stop_str)
print(struct_res)

/home/lcorti/miniconda3/envs/lmms/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


(Entity: Diamond Head Crater, Relationship: located in, Entity: Honolulu)
(Entity: ocean, Relationship: visible in, Entity: image)
(Entity: city, Relationship: located in, Entity: background)


In [73]:
# Get individual bullet points
items = struct_res.split('* ')
# Extract the structured information by
# (1) getting rid of everything before '(';
# (2) removing trailing spaces; and,
# (3) remove last ')' at the end.
relations = [i.split(' (')[1].rstrip()[:-1] for i in items if len(i) > 0]
for r in relations:
    print(r)

Entity: mountain, Relationship: has, Entity: flat top
Entity: mountain, Relationship: is near, Entity: body of water, Relationship: is a beach
Entity: mountain, Relationship: is surrounded by, Entity: city, Relationship: is in an urban area
Entity: city, Relationship: is densely populated, Entity: has numerous buildings, Entity: has numerous roads
Entity: sky, Relationship: is above, Entity: mountain, Relationship: has white clouds
Entity: photo, Relationship: is taken from, Entity: aerial perspective, Relationship: provides a bird's eye view
Entity: mountain, Relationship: is identified as, Entity: Diamond Head Crater, Relationship: is a well-known landmark
Entity: mountain, Relationship: is a popular tourist destination, Entity: has a beach, Entity: has a city
Entity: image, Relationship: captures, Entity: geological features, Relationship: is distinctive, Entity: has a shape, Entity: has a surrounding urban landscape


In [79]:
# Create individual relational triplets
def get_empty_triple():
    return {
        'from_concept': '',
        'rel_label': '',
        'to_concept': ''
    }

# == == Refine with POS tags??? == ==
triplets = []
for r in relations:
    elements = r.split(',')
    triplet = get_empty_triple()
    prev_k = ''
    for e in elements:
        k, v = e.split(': ')
        k = k.strip()
        v = v.strip()
        
        # Base cases
        if prev_k == '' and k == 'Entity':
            triplet['from_concept'] = v
            prev_k = k
        elif prev_k == 'Entity' and k == 'Relationship':
            triplet['rel_label'] = v
            prev_k = k
        elif prev_k == 'Relationship' and k == 'Entity':
            triplet['to_concept'] = v
            prev_k = k
            # The triple is complete, we can append it
            triplets.append(triplet)
            break


In [80]:
triplets

[{'from_concept': 'mountain', 'rel_label': 'has', 'to_concept': 'flat top'},
 {'from_concept': 'mountain',
  'rel_label': 'is near',
  'to_concept': 'body of water'},
 {'from_concept': 'mountain',
  'rel_label': 'is surrounded by',
  'to_concept': 'city'},
 {'from_concept': 'city',
  'rel_label': 'is densely populated',
  'to_concept': 'has numerous buildings'},
 {'from_concept': 'sky', 'rel_label': 'is above', 'to_concept': 'mountain'},
 {'from_concept': 'photo',
  'rel_label': 'is taken from',
  'to_concept': 'aerial perspective'},
 {'from_concept': 'mountain',
  'rel_label': 'is identified as',
  'to_concept': 'Diamond Head Crater'},
 {'from_concept': 'mountain',
  'rel_label': 'is a popular tourist destination',
  'to_concept': 'has a beach'},
 {'from_concept': 'image',
  'rel_label': 'captures',
  'to_concept': 'geological features'}]

In [22]:
# Ask follow-up question about explanation
struct_expl_q = "Carefully work through the list point by point and identify the grammatical subject, object, and predicate for each point in the list. "\
                "Fill in your response following this template (Subject: , Predicate:, Object: )."

conv = add_conv_step(conv, struct_expl_q, prev_resp=expl_res)
stop_str = get_stop_str(conv)
prompt = conv.get_prompt()

# Tokenize new input
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
stopping_criteria = get_stopping_criteria(input_ids, stop_str, tokenizer)
output_ids = gen_response(model, input_ids, image_tensor, gen_config, stopping_criteria)
struct_res = decode_tokens(input_ids, output_ids, tokenizer, stop_str)
print(struct_res)

* The image features a large, steep, and rocky mountain with a flat top. (Subject: mountain, Predicate: has, Object: a flat top)
* The mountain is situated near a body of water, which is the ocean. (Subject: mountain, Predicate: is situated, Object: near)
* The mountain is surrounded by a city, indicating that it is located in an urban area. (Subject: mountain, Predicate: is surrounded, Object: by)
* The city is densely populated, with numerous buildings and roads visible. (Subject: city, Predicate: is densely populated, Object: with)
* The mountain has a unique shape, resembling a diamond. (Subject: mountain, Predicate: has, Object: a unique shape)
* The name "Diamond Head" is associated with the mountain, suggesting that it is a well-known landmark. (Subject: mountain, Predicate: is associated, Object: with)
* The presence of the ocean and the city's location near the mountain indicate that this is a coastal area. (Subject: mountain, Predicate: is located, Object: in)
* The image is 

### Alt

In [9]:
expl_q = "Explain your previous answer by highlighting the concepts, and their relationships, from the image that lead you to your answer."

conv = add_conv_step(conv, expl_q, prev_resp=response)
stop_str = get_stop_str(conv)
prompt = conv.get_prompt()

# Tokenize new input
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
stopping_criteria = get_stopping_criteria(input_ids, stop_str, tokenizer)
output_ids = gen_response(model, input_ids, image_tensor, gen_config, stopping_criteria)
expl_res = decode_tokens(input_ids, output_ids, tokenizer, stop_str)
print(expl_res)

In the image, there is a large, circular mountain with a flat top, situated on the coast of a body of water. The mountain is surrounded by a city, indicating that it is a prominent landmark. The city appears to be built around the base of the mountain, suggesting that it is a popular tourist destination. The presence of a beach near the base of the mountain further supports this notion. The unique shape of the mountain, along with its location and the surrounding city, strongly resembles the Diamond Head Crater in Honolulu, Hawaii. This iconic landmark is known for its distinctive appearance and its historical significance as a military fortification during the early 20th century.


In [10]:
# Ask follow-up question about explanation
list_expl_q = "Create a bullet point list of the concepts and relationships you have just identified."

conv = add_conv_step(conv, list_expl_q, prev_resp=expl_res)
stop_str = get_stop_str(conv)
prompt = conv.get_prompt()

# Tokenize new input
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
stopping_criteria = get_stopping_criteria(input_ids, stop_str, tokenizer)
output_ids = gen_response(model, input_ids, image_tensor, gen_config, stopping_criteria)
struct_res = decode_tokens(input_ids, output_ids, tokenizer, stop_str)
print(struct_res)

Assistant: * The image features a large, circular mountain with a flat top.
* The mountain is situated on the coast of a body of water.
* The city is built around the base of the mountain, indicating that it is a popular tourist destination.
* A beach is located near the base of the mountain.
* The unique shape of the mountain, along with its location and the surrounding city, resembles the Diamond Head Crater in Honolulu, Hawaii.
* The Diamond Head Crater is a famous landmark known for its distinctive appearance and historical significance as a military fortification during the early 20th century.


In [11]:
# Ask follow-up question about explanation
struct_expl_q = "For each item in the list you just created, try to identify the subject, the object, and predicate."

conv = add_conv_step(conv, struct_expl_q, prev_resp=list_expl_q)
stop_str = get_stop_str(conv)
prompt = conv.get_prompt()

# Tokenize new input
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
stopping_criteria = get_stopping_criteria(input_ids, stop_str, tokenizer)
output_ids = gen_response(model, input_ids, image_tensor, gen_config, stopping_criteria)
struct_res = decode_tokens(input_ids, output_ids, tokenizer, stop_str)
print(struct_res)

Assistant: For each item in the list you just created, try to identify the subject, the object, and predicate.


### Dependency parsing way

Documentation here [https://spacy.io/api/dependencyparser](https://spacy.io/api/dependencyparser)

In [8]:
# Ask follow-up question about explanation
expl_q = "Explain your previous answer by highlighting the concepts, and their relationships, from the image that lead you to your answer."

conv = add_conv_step(conv, expl_q, prev_resp=response)
stop_str = get_stop_str(conv)
prompt = conv.get_prompt()

# Tokenize new input
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
stopping_criteria = get_stopping_criteria(input_ids, stop_str, tokenizer)
output_ids = gen_response(model, input_ids, image_tensor, gen_config, stopping_criteria)
response = decode_tokens(input_ids, output_ids, tokenizer, stop_str)
print(response)

The image features a large, brown, mountainous island with a crater in the center. The crater is surrounded by a beach and a body of water, which is likely the ocean. The island is situated in the middle of the ocean, and there are several smaller islands visible in the background. The sky above is blue, and there are clouds scattered across it. The overall scene suggests that this is a tropical location, possibly a popular tourist destination. The presence of the crater and the beach indicate that the island is likely volcanic in origin. The name "Diamond Head Crater" is associated with this type of geological formation, and it is known for its distinctive shape and appearance. Therefore, the image likely depicts the Diamond Head Crater in Hawaii.


In [9]:
import spacy
import csv
nlp = spacy.load('en_core_web_trf', exclude=['ner', 'lemmatizer', 'attribute_ruler'])
doc = nlp(response)

/home/lcorti/miniconda3/envs/lmms/lib/python3.12/site-packages/thinc/shims/pytorch.py:253: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(fil

In [10]:
nlp.pipeline

[('transformer',
  <spacy_curated_transformers.pipeline.transformer.CuratedTransformer at 0x7f8aaafeb9b0>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x7f8aaae128d0>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x7f8aaadfe340>)]

In [20]:
spacy_tags = ['ROOT', 'acl', 'acomp', 'advcl', 'advmod', 'agent', 'amod', 
              'appos', 'attr', 'aux', 'auxpass', 'case', 'cc', 'ccomp', 'compound', 
              'conj', 'csubj', 'csubjpass', 'dative', 'dep', 'det', 'dobj', 'expl', 
              'intj', 'mark', 'meta', 'neg', 'nmod', 'npadvmod', 'nsubj', 'nsubjpass', 
              'nummod', 'oprd', 'parataxis', 'pcomp', 'pobj', 'poss', 'preconj', 
              'predet', 'prep', 'prt', 'punct', 'quantmod', 'relcl', 'xcomp']

for st in spacy_tags:
    print('{}: {}'.format(st, spacy.explain(st)))

ROOT: root
acl: clausal modifier of noun (adjectival clause)
acomp: adjectival complement
advcl: adverbial clause modifier
advmod: adverbial modifier
agent: agent
amod: adjectival modifier
appos: appositional modifier
attr: attribute
aux: auxiliary
auxpass: auxiliary (passive)
case: case marking
cc: coordinating conjunction
ccomp: clausal complement
compound: compound
conj: conjunct
csubj: clausal subject
csubjpass: clausal subject (passive)
dative: dative
dep: unclassified dependent
det: determiner
dobj: direct object
expl: expletive
intj: interjection
mark: marker
meta: meta modifier
neg: negation modifier
nmod: modifier of nominal
npadvmod: noun phrase as adverbial modifier
nsubj: nominal subject
nsubjpass: nominal subject (passive)
nummod: numeric modifier
oprd: object predicate
parataxis: parataxis
pcomp: complement of preposition
pobj: object of preposition
poss: possession modifier
preconj: pre-correlative conjunction
predet: None
prep: prepositional modifier
prt: particle
punct

/home/lcorti/miniconda3/envs/lmms/lib/python3.12/site-packages/spacy/glossary.py:20: UserWarning: [W118] Term 'predet' not found in glossary. It may however be explained in documentation for the corpora used to train the language. Please check `nlp.meta["sources"]` for any relevant links.
  warnings.warn(Warnings.W118.format(term=term))


In [12]:
dependencies = []

for token in doc:
    dep_data = {}
    dep_data["tag"] = token.tag_
    dep_data["text"] = token.text
    dep_data["head"] = token.head.text
    dep_data["dep"] = token.dep_
    dep_data["dep_expl"] = spacy.explain(token.dep_)
    dependencies.append(dep_data)

    print(
        f"""
        TOKEN: {token.text}
        =====
        {token.tag_ = }
        {token.head.text = }
        {token.dep_ = }
        {spacy.explain(token.dep_)}"""
    )


        TOKEN: The
        =====
        token.tag_ = 'DT'
        token.head.text = 'image'
        token.dep_ = 'det'
        determiner

        TOKEN: image
        =====
        token.tag_ = 'NN'
        token.head.text = 'features'
        token.dep_ = 'nsubj'
        nominal subject

        TOKEN: features
        =====
        token.tag_ = 'VBZ'
        token.head.text = 'features'
        token.dep_ = 'ROOT'
        root

        TOKEN: a
        =====
        token.tag_ = 'DT'
        token.head.text = 'island'
        token.dep_ = 'det'
        determiner

        TOKEN: large
        =====
        token.tag_ = 'JJ'
        token.head.text = 'island'
        token.dep_ = 'amod'
        adjectival modifier

        TOKEN: ,
        =====
        token.tag_ = ','
        token.head.text = 'island'
        token.dep_ = 'punct'
        punctuation

        TOKEN: brown
        =====
        token.tag_ = 'JJ'
        token.head.text = 'island'
        token.dep_ = 'amod'
      

In [21]:
for token in doc:
    if token.pos_ in ['NOUN']:
        print('{} -- {}'.format(token, [t.text for t in token.subtree]))

In [ ]:
keys = dependencies[0].keys()
with open("dep_parse.csv", "w", newline='') as f:
    dict_writer = csv.DictWriter(f, keys)
    dict_writer.writeheader()
    dict_writer.writerows(dependencies)